# FedKG-Maint — Phase 5: KG-Grounded LLM Maintenance Explanations v1

**Goal:** Compare explanations for the same held-out-factory model alerts under two conditions: (A) without maintenance-KG evidence and (B) with timestamp-filtered facts retrieved from the local RDF graph.

**Inputs:** outputs from Phase 3 `phase3_federated_privacy_localnorm_v1/` and Phase 4 `phase4_local_knowledge_graphs_v1/`, plus the raw Azure telemetry table.

**Design controls:**
- The two conditions receive the same predicted risk score, machine metadata, and contemporaneous sensor readings. Only the graph condition receives retrieved historical error, maintenance, failure, and daily-summary facts.
- Retrieval filters every historical graph fact to `timestamp <= prediction timestamp`; future failure records must never be used to explain earlier predictions.
- The test-set label is used only for retrospective audit/sampling if needed; it is never placed in the LLM prompt.
- Citations are checked mechanically for whether IDs exist in the supplied evidence. That is **not** a semantic entailment proof. A human review sheet is generated for faithfulness/actionability annotation.
- The actual LLM comparison is not run automatically to avoid unexpected API calls. Add an API key via Colab Secrets/environment and explicitly turn `RUN_LLM_EXPERIMENT = True` to launch up to 24 calls (12 cases × 2 conditions).

The default API adapter is OpenAI-compatible. Set `OPENAI_MODEL`/`OPENAI_BASE_URL` if using a different endpoint/model. Never write an API key directly into notebook source.

In [ ]:
from pathlib import Path
import os, sys, json, time, warnings, subprocess, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    import rdflib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib"])
try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests
from rdflib import Graph, Namespace, RDF, RDFS

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
except ImportError:
    userdata = None

PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "/content/drive/MyDrive/FedKG-Maint")).resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
P3_DIR = PROJECT_DIR / "phase3_federated_privacy_localnorm_v1"
P4_DIR = PROJECT_DIR / "phase4_local_knowledge_graphs_v1"
OUT_DIR = PROJECT_DIR / "phase5_kg_llm_explanations_v1"
for p in ["inputs", "results", "reports", "logs"]: (OUT_DIR / p).mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Phase 3:", P3_DIR)
print("Phase 4:", P4_DIR)
print("Phase 5 output:", OUT_DIR)
assert (P4_DIR / "ontology" / "fedkg_shared_ontology.ttl").exists(), "Run Phase 4 graph build first."
assert (P3_DIR / "predictions" / "DP-FedAvg-CompactLinear__seed42_test_site_full.parquet").exists(), "Expected Phase 3 seed-42 held-out predictions not found."

In [ ]:
def locate_csv(*candidate_names):
    files = list(RAW_DIR.rglob("*.csv"))
    by_name = {p.name.casefold(): p for p in files}
    for name in candidate_names:
        if name.casefold() in by_name: return by_name[name.casefold()]
    raise FileNotFoundError(f"None of {candidate_names} found in {RAW_DIR}")

def normalize_id(value):
    if pd.isna(value): return None
    s = str(value).strip()
    try: return str(int(float(s)))
    except Exception: return s

PREP_DIR = PROJECT_DIR / "prep_v3"
sys.path.insert(0, str(PREP_DIR))
import importlib
sys.modules.pop("fedkg_data", None); importlib.invalidate_caches()
import fedkg_data

PRED_PATH = P3_DIR / "predictions" / "DP-FedAvg-CompactLinear__seed42_test_site_full.parquet"
pred = pd.read_parquet(PRED_PATH)
pred["machineID"] = pred["machineID"].map(normalize_id)
pred["datetime"] = pd.to_datetime(pred["datetime"], errors="coerce")
if not {"machineID", "datetime", "score"}.issubset(pred.columns):
    raise ValueError(f"Unexpected prediction columns: {list(pred.columns)}")

telemetry_path = locate_csv("PdM_telemetry.csv", "telemetry.csv")
telemetry = pd.read_csv(telemetry_path, low_memory=False)
if "machineID" not in telemetry.columns or "datetime" not in telemetry.columns:
    raise ValueError(f"Unexpected telemetry schema: {list(telemetry.columns)}")
telemetry["machineID"] = telemetry["machineID"].map(normalize_id)
telemetry["datetime"] = pd.to_datetime(telemetry["datetime"], errors="coerce")
sensors = [c for c in ["volt", "rotate", "pressure", "vibration"] if c in telemetry.columns]
if len(sensors) != 4: raise ValueError(f"Expected the four Azure telemetry fields; found {sensors}")
telemetry = telemetry.drop_duplicates(subset=["machineID", "datetime"])

X_test, y_test, meta_test = fedkg_data.load_split("test_site_full", policy="local", feature_set="full")
meta_keys = meta_test[["machineID", "datetime", "client_id"]].copy()
meta_keys["machineID"] = meta_keys["machineID"].map(normalize_id)
meta_keys["datetime"] = pd.to_datetime(meta_keys["datetime"], errors="coerce")
meta_keys["label_for_offline_audit_only"] = np.asarray(y_test, dtype=int)
joined = pred.merge(meta_keys, on=["machineID", "datetime"], how="left", validate="one_to_one")
if joined["client_id"].isna().any(): raise AssertionError("Predictions could not be matched to Phase 1 held-out metadata.")
if set(joined["client_id"].astype(str).unique()) != {"factory_5"}:
    raise AssertionError(f"Expected only held-out factory_5; found {joined['client_id'].unique()}")
joined = joined.sort_values("score", ascending=False).reset_index(drop=True)
print("Held-out predictions:", len(joined), "| prevalence (offline audit only):", round(joined["label_for_offline_audit_only"].mean(), 5))
print("Sensor columns:", sensors)
# Do not pass label_for_offline_audit_only to evidence retrieval or to either prompt.

In [ ]:
BASE = "https://example.org/fedkg-maint/"
ONT = Namespace(BASE + "ontology#")
RES = Namespace(BASE + "resource/")

def resource_uri(factory_id, entity_type, key):
    from urllib.parse import quote
    return RES[f"{quote(str(factory_id), safe='')}/{quote(str(entity_type), safe='')}/{quote(str(key), safe='')}"]

graph_path = P4_DIR / "graphs" / "factory_5.ttl"
kg = Graph()
kg.parse(str(graph_path), format="turtle")
print("Loaded held-out factory graph:", graph_path, "| triples:", len(kg))

def literal_float(g, subject, predicate):
    v = g.value(subject, predicate)
    try: return float(v) if v is not None else None
    except Exception: return None

def get_machine_static(g, machine_id):
    m = resource_uri("factory_5", "machine", normalize_id(machine_id))
    if (m, RDF.type, ONT.Machine) not in g: raise KeyError(f"machine {machine_id} absent from held-out KG")
    model_node = g.value(m, ONT.usesModel)
    model = str(g.value(model_node, ONT.modelName) or g.value(model_node, RDFS.label) or "unknown") if model_node else "unknown"
    age = literal_float(g, m, ONT.machineAge)
    return m, {"model": model, "age": age}

def _event_time(g, node):
    value = g.value(node, ONT.occurredAt)
    if value is None: return None
    try: return pd.Timestamp(str(value))
    except Exception: return None

def retrieve_kg_evidence(g, machine_id, as_of, lookback_days=30, telemetry_days=7, max_events_per_type=8):
    """Retrieve only graph facts whose source timestamps are <= the prediction time."""
    as_of = pd.Timestamp(as_of)
    m, static = get_machine_static(g, machine_id)
    facts = []
    input_id_counter = 1
    def add_fact(text, source_type, timestamp=None):
        facts.append({"evidence_id": f"KG-{len(facts)+1:03d}", "fact": text, "source_type": source_type,
                      "timestamp": timestamp.isoformat() if timestamp is not None else None})

    cutoff = as_of - pd.Timedelta(days=int(lookback_days))
    for pred, kind, related_pred in [
        (ONT.hasErrorOccurrence, "error", ONT.hasErrorCode),
        (ONT.hasMaintenanceEvent, "maintenance", ONT.targetsComponent),
        (ONT.hasFailureEvent, "failure", ONT.failureOfComponent),
    ]:
        rows = []
        for event_node in g.objects(m, pred):
            ts = _event_time(g, event_node)
            if ts is None or ts > as_of or ts < cutoff: continue
            related = g.value(event_node, related_pred)
            if kind == "error":
                value = str(g.value(related, ONT.errorCode) or g.value(related, RDFS.label) or "unknown error")
                phrase = f"An error occurrence with code {value} was recorded at {ts.isoformat()}"
            elif kind == "maintenance":
                value = str(g.value(related, ONT.componentName) or g.value(related, RDFS.label) or "unknown component")
                phrase = f"A maintenance record targeting component {value} was recorded at {ts.isoformat()}"
            else:
                value = str(g.value(related, ONT.componentName) or g.value(related, RDFS.label) or "unknown component")
                phrase = f"A failure record labeled {value} was observed at {ts.isoformat()}"
            rows.append((ts, phrase))
        for ts, phrase in sorted(rows, key=lambda z: z[0], reverse=True)[:max_events_per_type]:
            add_fact(phrase + ". This is historical association, not proof of cause.", kind, ts)

    # Daily telemetry facts are aggregates built in Phase 4; no future day is retrieved.
    tele_rows = []
    for node in g.objects(m, ONT.hasDailyTelemetry):
        dval = g.value(node, ONT.dailyDate)
        if dval is None: continue
        try: day = pd.Timestamp(str(dval))
        except Exception: continue
        if day.normalize() < as_of.normalize() and day.normalize() >= (as_of - pd.Timedelta(days=int(telemetry_days))).normalize():
            vals = {s: literal_float(g, node, getattr(ONT, f"{s}Mean")) for s in sensors}
            tele_rows.append((day, vals))
    for day, vals in sorted(tele_rows, key=lambda z: z[0], reverse=True)[:int(telemetry_days)]:
        pieces = [f"{s} mean={vals[s]:.4f}" for s in sensors if vals.get(s) is not None]
        if pieces: add_fact(f"Daily telemetry aggregate for {day.strftime('%Y-%m-%d')}: " + ", ".join(pieces) + ".", "daily_telemetry", day)

    # Safety invariant: every evidence record with a timestamp is at or before as_of.
    for fact in facts:
        if fact["timestamp"] is not None and pd.Timestamp(fact["timestamp"]) > as_of:
            raise AssertionError("Temporal leakage: retriever emitted future evidence")
    return static, facts

# Select 12 distinct risk-score cases across high, middle, and lower score ranges; labels are not used.
MAX_CASES = 12
n_top = min(6, MAX_CASES)
top_cases = joined.head(n_top)
remaining = joined.iloc[n_top:]
if len(remaining) > 0:
    qlo, qhi = remaining["score"].quantile([0.20, 0.80]).tolist()
    middle = remaining[(remaining["score"] >= qlo) & (remaining["score"] <= qhi)]
    n_other = min(MAX_CASES - len(top_cases), 6)
    other_cases = middle.sample(n=min(n_other, len(middle)), random_state=2026) if len(middle) else remaining.sample(n=min(n_other, len(remaining)), random_state=2026)
    cases = pd.concat([top_cases, other_cases]).drop_duplicates(subset=["machineID", "datetime"]).head(MAX_CASES)
else:
    cases = top_cases.copy()
cases = cases.reset_index(drop=True)
print("Explanation cases selected by risk-score spread (no labels provided to prompts):", len(cases))
display(cases[["machineID", "datetime", "score", "client_id"]].head(12))

In [ ]:
# Build fixed inputs and KG evidence packs. The offline labels are excluded from all prompt payloads.
case_payloads = []
telemetry_lookup = telemetry.set_index(["machineID", "datetime"], drop=False)
for case_idx, row in cases.iterrows():
    mid, ts = row["machineID"], pd.Timestamp(row["datetime"])
    machine_rows = telemetry[(telemetry["machineID"] == mid) & (telemetry["datetime"] == ts)]
    if len(machine_rows) == 0:
        # Exact row normally exists; nearest prior record is explicitly labeled if timestamp matching differs.
        prior = telemetry[(telemetry["machineID"] == mid) & (telemetry["datetime"] <= ts)].sort_values("datetime")
        if prior.empty: raise AssertionError(f"No telemetry input available for machine={mid}, time={ts}")
        sensor_row = prior.iloc[-1]
        sensor_timestamp = pd.Timestamp(sensor_row["datetime"])
        sensor_note = "nearest_prior_raw_telemetry"
    else:
        sensor_row = machine_rows.iloc[0]
        sensor_timestamp = ts
        sensor_note = "exact_timestamp_raw_telemetry"
    current_sensors = {s: float(sensor_row[s]) for s in sensors}
    static, kg_facts = retrieve_kg_evidence(kg, mid, ts, lookback_days=30, telemetry_days=7)
    base_facts = [
        {"evidence_id": "INPUT-001", "fact": f"The model's predicted failure-risk score for this machine-time row is {float(row['score']):.6f}.", "source_type": "model_output", "timestamp": ts.isoformat()},
        {"evidence_id": "INPUT-002", "fact": f"Machine {mid} current raw telemetry ({sensor_note}, timestamp {sensor_timestamp.isoformat()}): " + ", ".join(f"{k}={v:.4f}" for k, v in current_sensors.items()) + ".", "source_type": "current_telemetry", "timestamp": sensor_timestamp.isoformat()},
    ]
    if static.get("model") != "unknown": base_facts.append({"evidence_id": "INPUT-003", "fact": f"Inventory records machine model {static['model']}.", "source_type": "machine_inventory", "timestamp": None})
    if static.get("age") is not None: base_facts.append({"evidence_id": "INPUT-004", "fact": f"Inventory records machine age {static['age']:g}.", "source_type": "machine_inventory", "timestamp": None})
    # Phase 5 only sends the risk score/context and as-of evidence; labels are never included.
    case_payloads.append({
        "case_id": f"CASE-{case_idx+1:03d}", "factory": "factory_5", "machineID": mid,
        "prediction_timestamp": ts.isoformat(), "risk_score": float(row["score"]),
        "current_sensor_values": current_sensors, "sensor_timestamp": sensor_timestamp.isoformat(),
        "base_facts": base_facts, "kg_facts": kg_facts,
        "offline_label_for_evaluation_only": int(row["label_for_offline_audit_only"]),
    })

packs_path = OUT_DIR / "inputs" / "explanation_case_evidence_packs.jsonl"
with packs_path.open("w") as f:
    for item in case_payloads:
        f.write(json.dumps({k: v for k, v in item.items() if k != "offline_label_for_evaluation_only"}, default=str) + "\n")
print("Evidence packs saved:", packs_path)
print("Mean KG facts per case:", round(np.mean([len(x["kg_facts"]) for x in case_payloads]), 2))
print("Temporal safety:", "all KG evidence timestamps checked <= prediction timestamp")

In [ ]:
# API config. Pull secrets only from environment or Colab Secrets; never print the key.
API_KEY = os.environ.get("OPENAI_API_KEY")
if not API_KEY and userdata is not None:
    try: API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception: API_KEY = None
API_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
LLM_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
RUN_LLM_EXPERIMENT = False  # Set True deliberately after confirming API access/cost.

SYSTEM_PROMPT = """You are an industrial maintenance assistant explaining a predictive-maintenance model alert. Do not claim certainty about future failure. Do not assert causation from historical co-occurrence. Distinguish direct observations from hypotheses. Recommend safe inspection/diagnostic steps rather than asserting a component is broken. Return only a JSON object with keys: summary (string), claims (array of objects with text and evidence_ids array), recommended_actions (array of strings), uncertainty (array of strings). Cite each factual claim using only evidence IDs supplied in the user message. Do not invent evidence IDs. If information is insufficient, say so explicitly."""

def make_user_prompt(payload, with_kg):
    available = list(payload["base_facts"])
    condition = "WITHOUT_KG"
    if with_kg:
        available += payload["kg_facts"]
        condition = "WITH_KG"
    data = {
        "condition": condition,
        "factory": payload["factory"],
        "machineID": payload["machineID"],
        "prediction_timestamp": payload["prediction_timestamp"],
        "note": "Only facts listed below are available. Event facts are timestamp-filtered to this prediction time. This is a risk score, not a calibrated probability unless separately verified.",
        "available_facts": available,
    }
    return "Explain this predictive-maintenance alert using the available facts. Do not describe the target label or claim the machine actually failed. Keep the explanation concise and distinguish evidence from possible interpretation.\n\n" + json.dumps(data, ensure_ascii=False)

def call_llm(payload, with_kg, timeout=90):
    if not API_KEY: raise RuntimeError("No API key found. Add OPENAI_API_KEY through Colab Secrets/environment; do not hardcode it.")
    url = API_BASE_URL.rstrip("/") + "/chat/completions"
    body = {
        "model": LLM_MODEL,
        "temperature": 0.1,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_user_prompt(payload, with_kg)},
        ],
    }
    resp = requests.post(url, headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, json=body, timeout=timeout)
    if not resp.ok:
        raise RuntimeError(f"LLM endpoint returned HTTP {resp.status_code}: {resp.text[:1000]}")
    content = resp.json()["choices"][0]["message"]["content"]
    try: parsed = json.loads(content)
    except Exception:
        match = re.search(r"\{.*\}", content, flags=re.S)
        if not match: raise ValueError("LLM did not return a JSON object; raw output saved in exception.")
        parsed = json.loads(match.group(0))
    for k, default in [("summary", ""), ("claims", []), ("recommended_actions", []), ("uncertainty", [])]:
        if k not in parsed: parsed[k] = default
    return parsed

def score_citations(parsed, allowed_ids):
    claims = parsed.get("claims", []) or []
    valid_ids = set(allowed_ids)
    cited_claims = 0
    invalid_ids = []
    claim_count = 0
    for claim in claims:
        if not isinstance(claim, dict): continue
        claim_count += 1
        ids = claim.get("evidence_ids", []) or []
        if isinstance(ids, str): ids = [ids]
        if any(x in valid_ids for x in ids): cited_claims += 1
        invalid_ids.extend([x for x in ids if x not in valid_ids])
    return {"claim_count": claim_count,
            "claims_with_at_least_one_valid_id": cited_claims,
            "valid_citation_claim_ratio": cited_claims / claim_count if claim_count else np.nan,
            "invalid_citation_count": len(invalid_ids),
            "invalid_citation_ids": sorted(set(map(str, invalid_ids)))}

print({"api_key_available": bool(API_KEY), "api_base_url": API_BASE_URL, "model": LLM_MODEL,
       "run_llm_experiment": RUN_LLM_EXPERIMENT, "planned_calls": len(case_payloads)*2})
if RUN_LLM_EXPERIMENT and not API_KEY:
    raise RuntimeError("RUN_LLM_EXPERIMENT=True but OPENAI_API_KEY is not configured.")

In [ ]:
# Optional paired ablation: same cases, same base context, graph history is the only input difference.
llm_rows = []
raw_outputs = []
if RUN_LLM_EXPERIMENT:
    for idx, payload in enumerate(case_payloads, start=1):
        for with_kg in [False, True]:
            condition = "with_kg" if with_kg else "without_kg"
            print(f"Calling LLM {idx}/{len(case_payloads)} — {payload['case_id']} — {condition}")
            try:
                parsed = call_llm(payload, with_kg)
                available = payload["base_facts"] + (payload["kg_facts"] if with_kg else [])
                citation_metrics = score_citations(parsed, [f["evidence_id"] for f in available])
                row = {
                    "case_id": payload["case_id"], "factory": payload["factory"], "machineID": payload["machineID"],
                    "prediction_timestamp": payload["prediction_timestamp"], "risk_score": payload["risk_score"],
                    "condition": condition, "status": "OK", "summary": parsed.get("summary", ""),
                    "claims_json": json.dumps(parsed.get("claims", []), ensure_ascii=False),
                    "recommended_actions_json": json.dumps(parsed.get("recommended_actions", []), ensure_ascii=False),
                    "uncertainty_json": json.dumps(parsed.get("uncertainty", []), ensure_ascii=False),
                    "kg_fact_count_supplied": len(payload["kg_facts"]) if with_kg else 0,
                    **citation_metrics,
                }
                llm_rows.append(row)
                raw_outputs.append({"case_id": payload["case_id"], "condition": condition, "output": parsed})
            except Exception as e:
                llm_rows.append({"case_id": payload["case_id"], "factory": payload["factory"], "machineID": payload["machineID"],
                                 "prediction_timestamp": payload["prediction_timestamp"], "risk_score": payload["risk_score"],
                                 "condition": condition, "status": "ERROR", "error": repr(e)})
            time.sleep(0.2)
    results = pd.DataFrame(llm_rows)
    results.to_csv(OUT_DIR / "results" / "llm_explanation_ablation.csv", index=False)
    with (OUT_DIR / "results" / "llm_explanation_outputs.jsonl").open("w") as f:
        for r in raw_outputs: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Saved LLM results:", OUT_DIR / "results" / "llm_explanation_ablation.csv")
    display(results[[c for c in ["case_id", "condition", "status", "claim_count", "valid_citation_claim_ratio", "invalid_citation_count"] if c in results.columns]].head(12))
else:
    print("LLM calls skipped by design. Evidence packs are ready. To run the actual comparison, set RUN_LLM_EXPERIMENT=True in this cell after confirming API access/cost.")
    results = pd.DataFrame()

In [ ]:
# Produce a human-review template regardless of whether the model call was run.
conditions = ["without_kg", "with_kg"]
review_rows = []
for payload in case_payloads:
    for condition in conditions:
        review_rows.append({
            "case_id": payload["case_id"], "condition": condition,
            "prediction_timestamp": payload["prediction_timestamp"], "machineID": payload["machineID"],
            "faithfulness_1to5": "", "evidence_support_1to5": "", "actionability_1to5": "",
            "unsupported_claim_count": "", "unsafe_or_overconfident_advice": "", "reviewer_notes": ""
        })
review_df = pd.DataFrame(review_rows)
review_path = OUT_DIR / "reports" / "human_evaluation_template.csv"
review_df.to_csv(review_path, index=False)

report = {
    "phase": "Phase 5 — KG-Grounded LLM Maintenance Explanations v1",
    "cases_prepared": len(case_payloads),
    "conditions": ["without_kg", "with_kg"],
    "kg_timestamp_guard": "All timestamped KG evidence is checked to be <= prediction timestamp.",
    "test_labels_sent_to_llm": False,
    "llm_experiment_requested": bool(RUN_LLM_EXPERIMENT),
    "api_key_available": bool(API_KEY),
    "llm_calls_planned": len(case_payloads) * 2,
    "llm_calls_completed_ok": int((results.get("status", pd.Series(dtype=str)) == "OK").sum()) if not results.empty else 0,
    "llm_calls_failed": int((results.get("status", pd.Series(dtype=str)) == "ERROR").sum()) if not results.empty else 0,
    "human_review_template": str(review_path),
    "limitations": [
        "A valid evidence ID shows that a supplied source exists, not that every sentence is semantically entailed by it.",
        "Human review or a separately validated entailment evaluator is required before claiming lower hallucination rate.",
        "API availability/model support and cost depend on the selected endpoint/account.",
        "The benchmark held-out set was used in earlier project analyses; this is not a new untouched external test.",
        "Graph facts reflect observed records; historical co-occurrence is not causal proof."
    ]
}
if RUN_LLM_EXPERIMENT and not results.empty:
    report["condition_summary"] = results[results["status"] == "OK"].groupby("condition").agg(
        cases=("case_id", "nunique"), mean_valid_citation_claim_ratio=("valid_citation_claim_ratio", "mean"),
        mean_invalid_citation_count=("invalid_citation_count", "mean")).reset_index().to_dict("records")
(OUT_DIR / "reports" / "phase5_llm_explanation_report.json").write_text(json.dumps(report, indent=2, default=str))
md = [
    "# Phase 5 — KG-Grounded LLM Maintenance Explanations v1", "",
    f"- Explanation cases prepared: **{len(case_payloads)}**.",
    f"- LLM ablation actually run: **{bool(RUN_LLM_EXPERIMENT)}**.",
    f"- API key available: **{bool(API_KEY)}** (key value never written to report).",
    f"- Successful model calls: **{report['llm_calls_completed_ok']}**; failed calls: **{report['llm_calls_failed']}**.",
    "- Test labels sent to the LLM: **No**.",
    "- Human review template: `reports/human_evaluation_template.csv`.",
    "", "## Interpretation limits", *[f"- {x}" for x in report["limitations"]],
    "", "## Decision", ("LLM ablation executed; review saved outputs and complete the human evaluation before making quality claims." if RUN_LLM_EXPERIMENT else "Evidence retrieval and paired case inputs are prepared. LLM comparison remains **not run** until explicitly enabled with a valid API configuration.")
]
(OUT_DIR / "reports" / "phase5_llm_explanation_report.md").write_text("\n".join(md))
print("\n".join(md))
print("Report saved to:", OUT_DIR / "reports" / "phase5_llm_explanation_report.md")

## What counts as completion

Do not declare the LLM phase successful merely because the notebook runs. At minimum: (1) verify evidence packs obey the timestamp guard; (2) run both conditions on the same cases; (3) inspect API errors; (4) have a reviewer annotate the generated `human_evaluation_template.csv`; and (5) report citation-validity separately from semantic factuality. If no API key/model is available, report the retrieval artifacts and mark the LLM experiment as not run.